# Gold booking revenue, daily
Reference output for Prompt 2b-1 once the instructions and skills are on (`de-job-standards`, `table-governance`, `data-quality-checks`). Follows `project/docs/project_context.md`.

Writes `{catalog}.{schema}.gold_booking_revenue_daily`: one row per `check_in_date`, `destination_id` and `property_type`, from `samples.wanderbricks.bookings` joined to `properties` and `destinations`.

* **Cancelled bookings are excluded** from `net_revenue_amount`, `booking_count` and `avg_booking_value`; `cancelled_booking_count` and `cancelled_amount` are kept so the exclusion is visible.
* Full overwrite on every run (project decision: small data, simpler than MERGE).
* Reads `bookings.status` directly; the `silver_bookings` step (latest status from `booking_updates`) is left out of this reference.
* Task of the `wanderbricks_booking_revenue_daily` job (tag `project:asml_demo`).

In [ ]:
# Target: {catalog}.{schema}.gold_booking_revenue_daily (PK: check_in_date, destination_id, property_type)
dbutils.widgets.text("catalog", "uaigw_fe")
dbutils.widgets.text("schema", "asml_demo")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
target = f"{catalog}.{schema}.gold_booking_revenue_daily"
print(target)

In [ ]:
from pyspark.sql import functions as F

bookings = spark.table("samples.wanderbricks.bookings")
properties = spark.table("samples.wanderbricks.properties").select("property_id", "destination_id", "property_type")
destinations = spark.table("samples.wanderbricks.destinations").select(
    "destination_id", "destination", F.col("country").alias("destination_country")
)

is_cancelled = F.col("status") == "cancelled"

df = (
    bookings.join(properties, "property_id", "inner")
    .join(destinations, "destination_id", "inner")
    .groupBy(F.col("check_in").alias("check_in_date"), "destination_id", "property_type", "destination", "destination_country")
    .agg(
        F.sum(F.when(~is_cancelled, 1).otherwise(0)).alias("booking_count"),
        F.sum(F.when(is_cancelled, 1).otherwise(0)).alias("cancelled_booking_count"),
        F.sum(F.when(~is_cancelled, F.datediff("check_out", "check_in")).otherwise(0)).alias("booked_nights"),
        F.round(F.sum(F.when(~is_cancelled, F.col("total_amount")).otherwise(0)), 2).alias("net_revenue_amount"),
        F.round(F.sum(F.when(is_cancelled, F.col("total_amount")).otherwise(0)), 2).alias("cancelled_amount"),
    )
    .withColumn(
        "avg_booking_value",
        F.round(F.col("net_revenue_amount") / F.nullif(F.col("booking_count"), F.lit(0)), 2),
    )
    .withColumn("audit_timestamp", F.current_timestamp())
    .withColumn("source_system", F.lit("samples.wanderbricks"))
    # keys -> dimensions -> measures -> derived -> audit columns LAST
    .select(
        "check_in_date", "destination_id", "property_type",
        "destination", "destination_country",
        "booking_count", "cancelled_booking_count", "booked_nights",
        "net_revenue_amount", "cancelled_amount", "avg_booking_value",
        "audit_timestamp", "source_system",
    )
)

if df.limit(1).count() == 0:
    raise ValueError(f"No rows produced for {target}")

In [ ]:
# Full overwrite: re-runs replace the table, never duplicate rows
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target)
print(f"{spark.table(target).count()} rows in {target}")

In [ ]:
# DDL per table-governance standard; re-applied on every run because overwriteSchema resets comments
spark.sql(f"""COMMENT ON TABLE {target} IS
  'Business aggregation: daily booking revenue by check-in date, destination and property type from samples.wanderbricks.bookings. Cancelled bookings excluded from net revenue.'""")

spark.sql(f"""ALTER TABLE {target} SET TBLPROPERTIES (
  'quality' = 'gold',
  'domain' = 'bookings',
  'contains_pii' = 'false',
  'delta.enableChangeDataFeed' = 'true')""")

column_comments = {
    "check_in_date": "Check-in date of the booking (bookings.check_in). Part of the primary key",
    "destination_id": "Part of the primary key. References samples.wanderbricks.destinations.destination_id",
    "property_type": "Part of the primary key. Property type from samples.wanderbricks.properties",
    "destination": "Destination name, from samples.wanderbricks.destinations",
    "destination_country": "Destination country, from samples.wanderbricks.destinations.country",
    "booking_count": "COUNT of non-cancelled bookings",
    "cancelled_booking_count": "COUNT of bookings with status = cancelled (excluded from revenue)",
    "booked_nights": "SUM(check_out - check_in) in nights over non-cancelled bookings",
    "net_revenue_amount": "SUM(total_amount) over non-cancelled bookings. Units: currency of total_amount",
    "cancelled_amount": "SUM(total_amount) over cancelled bookings, excluded from net_revenue_amount",
    "avg_booking_value": "Calculated as net_revenue_amount / booking_count, null when booking_count = 0",
    "audit_timestamp": "Job execution timestamp",
    "source_system": "Upstream source system identifier",
}
for col, comment in column_comments.items():
    spark.sql(f"ALTER TABLE {target} ALTER COLUMN {col} COMMENT '{comment}'")

# Governed tag policies differ per account (e.g. data_classification may only allow certain values).
# Set each tag on its own and skip any the policy rejects; domain and PII status are in TBLPROPERTIES.
for key, value in {"quality": "gold", "data_classification": "internal", "sla": "daily"}.items():
    try:
        spark.sql(f"ALTER TABLE {target} SET TAGS ('{key}' = '{value}')")
    except Exception as e:
        if "TAG_POLICY" not in str(e):
            raise
        print(f"Skipped tag {key}={value}: not allowed by this account's tag policy")

In [ ]:
# Copy of asml-workshop/skills/job-standards/scripts/check_table_standards.py (metadata reads only)
import re

LAYER_PREFIXES = ("bronze_", "silver_", "gold_")
REQUIRED_PROPS = ("quality", "domain")
AUDIT_COLUMNS = ["audit_timestamp", "source_system"]
NAME_PATTERN = re.compile(r"^[a-z][a-z0-9_]*$")

def check_table_standards(full_name: str) -> list:
    failures = []
    table = full_name.split(".")[2]
    if not NAME_PATTERN.match(table):
        failures.append(f"Table name '{table}' is not lowercase_snake_case")
    if not table.startswith(LAYER_PREFIXES):
        failures.append(f"Table name '{table}' has no layer prefix {LAYER_PREFIXES}")
    if not spark.sql(f"DESCRIBE DETAIL {full_name}").first()["description"]:
        failures.append("Missing table COMMENT")
    props = {r["key"]: r["value"] for r in spark.sql(f"SHOW TBLPROPERTIES {full_name}").collect()}
    failures += [f"Missing TBLPROPERTY '{p}'" for p in REQUIRED_PROPS if p not in props]
    if props.get("contains_pii") == "true" and not props.get("pii_columns"):
        failures.append("contains_pii=true but 'pii_columns' is not set")
    fields = spark.table(full_name).schema.fields
    if [f.name for f in fields][-2:] != AUDIT_COLUMNS:
        failures.append(f"Last two columns must be {AUDIT_COLUMNS}")
    undocumented = [f.name for f in fields if not f.metadata.get("comment")]
    if undocumented:
        failures.append(f"Columns without COMMENT: {undocumented}")
    print(f"[{'PASS' if not failures else f'FAIL ({len(failures)})'}] {full_name}")
    for f in failures:
        print(f"  - {f}")
    return failures

failures = check_table_standards(target)

t = spark.table(target)
# Primary key must be unique
dupes = t.groupBy("check_in_date", "destination_id", "property_type").count().filter("count > 1").count()
if dupes:
    failures.append(f"{dupes} duplicate (check_in_date, destination_id, property_type) rows")
# Reconcile with the source: net revenue and cancelled amount must match the bookings table
src = spark.sql("""SELECT round(sum(total_amount) FILTER (WHERE status <> 'cancelled'), 0) AS net,
                          round(sum(total_amount) FILTER (WHERE status = 'cancelled'), 0) AS cancelled
                   FROM samples.wanderbricks.bookings""").first()
tgt = t.agg(F.round(F.sum("net_revenue_amount"), 0).alias("net"), F.round(F.sum("cancelled_amount"), 0).alias("cancelled")).first()
for k in ("net", "cancelled"):
    if abs(src[k] - tgt[k]) > 1:
        failures.append(f"{k} total {tgt[k]} does not reconcile with source {src[k]}")

if failures:
    raise AssertionError(f"{target} does not meet standards: {failures}")

display(t.groupBy("property_type").agg(F.sum("net_revenue_amount").alias("net_revenue_amount"), F.sum("cancelled_amount").alias("cancelled_amount")).orderBy(F.desc("net_revenue_amount")))